# Export embeddings and land cover

Run `starter.ipynb` first to save `outputs/model.pt`. This notebook writes the middle-layer embeddings as float32 and class predictions as uint8 COGs on the source grid. Ocean and excluded labels are masked; lakes are included.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neuralearthfields/neuralearthfields-starter/blob/main/inference.ipynb)

## Setup

Locally, run `uv sync` and open this notebook with `uv run jupyter lab`. In Colab, run the cell below to download the helper code and install dependencies. Select a GPU runtime for training. If Colab requests a runtime restart after installation, restart before continuing.

In [1]:
import os
import subprocess
import sys
import tomllib
from pathlib import Path

if "google.colab" in sys.modules:
    repo = Path("/content/neuralearthfields-starter")
    if not repo.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/neuralearthfields/neuralearthfields-starter.git",
                str(repo),
            ],
            check=True,
        )
    os.chdir(repo)
    sys.path.insert(0, str(repo))
    dependencies = tomllib.loads(Path("pyproject.toml").read_text())["project"][
        "dependencies"
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", *dependencies], check=True)

In [2]:
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import rasterio
import torch
from rasterio.shutil import copy as raster_copy
from rasterio.windows import Window
from torch import nn


class Wrap(nn.Module):
    def forward(self, coords):
        coords.deg2rad_()
        features = coords.repeat_interleave(2, dim=-1)
        features[:, 0::2].cos_()
        features[:, 1::2].sin_()
        return features


def load_model(checkpoint, device):
    saved = torch.load(checkpoint, map_location=device, weights_only=True)
    encoder = nn.Sequential(
        Wrap(),
        nn.Linear(4, saved["input_dim"]),
        nn.ReLU(inplace=True),
        nn.Linear(saved["input_dim"], saved["hidden_dim"]),
        nn.ReLU(inplace=True),
    )
    decoder = nn.Sequential(
        nn.Linear(saved["hidden_dim"], saved["output_dim"]), nn.ReLU(inplace=True)
    )
    model = nn.Sequential(
        encoder, decoder, nn.Linear(saved["output_dim"], saved["num_classes"])
    )
    model = model.to(device)
    model.load_state_dict(saved["model"])
    return model.eval(), np.asarray(saved["class_codes"], dtype="uint8")

## Load the checkpoint

Inference uses plain PyTorch. Change the device or batch size for your hardware.

In [3]:
source = Path("data/CGLC-MODIS-LCZ-100m.cog.tif")
checkpoint = Path("outputs/model.pt")
output_dir = Path("outputs/cogs")
device = "cpu"
batch_size = 4096
write_global_cogs = False
torch.set_num_threads(4)
model, class_codes = load_model(checkpoint, device)
print(model)

Sequential(
  (0): Sequential(
    (0): Wrap()
    (1): Linear(in_features=4, out_features=256, bias=True)
    (2): ReLU(inplace=True)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU(inplace=True)
  )
  (1): Sequential(
    (0): Linear(in_features=128, out_features=256, bias=True)
    (1): ReLU(inplace=True)
  )
  (2): Linear(in_features=256, out_features=31, bias=True)
)


## Write COGs

The default export covers the entire source grid. The 128-band embedding raster contains about 31.5 TB before compression, plus space for temporary rasters and overviews. Choose an output disk and set `write_global_cogs = True` to run. An optional `Window(col, row, width, height)` lets you check a subset.

Embedding overviews use averages; class overviews use modes. The source grid is nominally 100 m in EPSG:4326, so ground spacing varies with latitude.

### Predict one tile

Build pixel-center coordinates, run the model in batches, and return embedding bands and class IDs. Excluded pixels remain NaN in embeddings and zero in predictions.

In [4]:
def pixel_coordinates(transform, window):
    rows, cols = np.indices((int(window.height), int(window.width)))
    lon, lat = transform * (cols + window.col_off + 0.5, rows + window.row_off + 0.5)
    return np.column_stack((lon.ravel(), lat.ravel())).astype("float32")


@torch.inference_mode()
def predict_tile(model, class_codes, coords, valid, batch_size=4096):
    model.eval()
    device = next(model.parameters()).device
    encoder, head = model[0], model[1:]
    bands = model[1][0].in_features
    features = np.full((len(coords), bands), np.nan, dtype="float32")
    classes = np.zeros(len(coords), dtype="uint8")
    valid_pixels = np.flatnonzero(valid.ravel())
    for start in range(0, len(valid_pixels), batch_size):
        selected = valid_pixels[start : start + batch_size]
        inputs = torch.from_numpy(coords[selected]).to(device)
        embeddings = encoder(inputs)
        features[selected] = embeddings.float().cpu().numpy()
        classes[selected] = class_codes[head(embeddings).argmax(1).cpu().numpy()]
    return features.T.reshape(bands, *valid.shape), classes.reshape(valid.shape)

### Write tiled rasters

Use the source grid and embedded palette. Read, predict, and write one tile at a time. Band-interleaved staging keeps embedding overview generation efficient.

In [5]:
def raster_profile(src, window):
    return {
        "driver": "GTiff",
        "crs": src.crs,
        "transform": src.window_transform(window),
        "width": int(window.width),
        "height": int(window.height),
        "tiled": True,
        "blockxsize": 512,
        "blockysize": 512,
        "compress": "zstd",
        "zstd_level": 3,
        "BIGTIFF": "YES",
        "NUM_THREADS": "4",
    }


def write_tiles(model, class_codes, src, window, embeddings, predictions, batch_size):
    colors = src.colormap(1)
    colors[0] = (0, 0, 0, 0)
    predictions.write_colormap(1, colors)
    predictions.set_band_description(1, "Land cover class ID")
    for band in range(embeddings.count):
        embeddings.set_band_description(band + 1, f"embedding_{band}")
    for index, (_, tile) in enumerate(embeddings.block_windows(1), 1):
        source_tile = Window(
            window.col_off + tile.col_off,
            window.row_off + tile.row_off,
            tile.width,
            tile.height,
        )
        labels = src.read(1, window=source_tile, masked=True)
        valid = np.isin(labels.data, class_codes) & ~np.ma.getmaskarray(labels)
        coords = pixel_coordinates(src.transform, source_tile)
        features, classes = predict_tile(model, class_codes, coords, valid, batch_size)
        embeddings.write(features, window=tile)
        predictions.write(classes, 1, window=tile)
        if index % 100 == 0:
            print(f"{index:,} tiles written", flush=True)


def write_rasters(model, class_codes, src, window, directory, batch_size):
    embedding_path = Path(directory) / "embeddings.tif"
    prediction_path = Path(directory) / "predictions.tif"
    profile = raster_profile(src, window)
    with (
        rasterio.open(
            embedding_path,
            "w",
            **profile,
            count=model[1][0].in_features,
            dtype="float32",
            nodata=np.nan,
            predictor=3,
            interleave="band",
        ) as embeddings,
        rasterio.open(
            prediction_path,
            "w",
            **profile,
            count=1,
            dtype="uint8",
            nodata=0,
            predictor=2,
        ) as predictions,
    ):
        write_tiles(
            model, class_codes, src, window, embeddings, predictions, batch_size
        )
    return embedding_path, prediction_path

### Convert to COG

`convert_to_cog` adds overviews and COG layout to a raster. `export_cogs` connects the steps and manages temporary files. Pass whole-pixel windows within the source and a positive batch size.

In [6]:
def convert_to_cog(source, destination, resampling, predictor):
    raster_copy(
        source,
        destination,
        driver="COG",
        COMPRESS="ZSTD",
        LEVEL=3,
        BLOCKSIZE=512,
        BIGTIFF="YES",
        NUM_THREADS="4",
        PREDICTOR=predictor,
        OVERVIEW_RESAMPLING=resampling,
    )
    return destination


def export_cogs(model, class_codes, source, output_dir, window=None, batch_size=4096):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    class_codes = np.asarray(class_codes, dtype="uint8")
    with rasterio.open(source) as src, TemporaryDirectory(dir=output_dir) as temporary:
        window = window or Window(0, 0, src.width, src.height)
        staged_embeddings, staged_predictions = write_rasters(
            model, class_codes, src, window, temporary, batch_size
        )
        embeddings = convert_to_cog(
            staged_embeddings,
            output_dir / "embeddings_100m.cog.tif",
            "AVERAGE",
            "FLOATING_POINT",
        )
        predictions = convert_to_cog(
            staged_predictions,
            output_dir / "land_cover_100m.cog.tif",
            "MODE",
            "STANDARD",
        )
    return embeddings, predictions

In [7]:
if write_global_cogs:
    embedding_path, prediction_path = export_cogs(
        model, class_codes, source, output_dir, batch_size=batch_size
    )
    print(embedding_path, prediction_path)

## Read embeddings

The bands are the middle bottleneck features, before expansion. Apply the saved linear probe directly to them.

In [8]:
if write_global_cogs:
    saved = torch.load(checkpoint, map_location="cpu", weights_only=True)
    head = nn.Linear(saved["hidden_dim"], saved["num_classes"])
    head.load_state_dict(saved["linear_head"])
    with rasterio.open(embedding_path) as src:
        tile = src.read(window=Window(0, 0, 256, 256))
    features = torch.from_numpy(tile.reshape(saved["hidden_dim"], -1).T.copy())
    valid = torch.isfinite(features).all(1)
    prediction = np.zeros(len(features), dtype="uint8")
    with torch.inference_mode():
        prediction[valid.numpy()] = class_codes[head(features[valid]).argmax(1).numpy()]
    prediction = prediction.reshape(tile.shape[1:])